# Lab 04 — DPO From Scratch (prototype notebook)

**Companion to `starter.py`** — prototype here (small runs, plotting); export the training loop to a script for cluster runs. Both read `configs/04_dpo_from_scratch.yaml`.

Prereqs: Lab 03 done · [Lecture 6](https://rlhfbook.com/course) · [Chapter 8](https://rlhfbook.com/c/08-direct-alignment.html).

> Everything in `starter.py` is yours to implement. The rlhf-book repo (`direct_alignment/loss.py::DPOLoss`, `data.py`, `configs/dpo.yaml`) is the **answer key** — open it only after your version trains.

## 0. Setup (Colab)

Run once per session; no other installs needed for CPU-side verification of Parts 1–2.

In [ ]:
# TODO: %pip install torch transformers datasets matplotlib
# TODO: load config (configs/04_dpo_from_scratch.yaml) into CONFIG dict
# TODO: set seed protocol from config['seed']

## 1. Paired data → PreferenceBatch

Build on your Lab 02 cleaned subset (1–3k pairs). The invariant that matters:
`response_mask` = 1 ONLY at response/EOS positions; prompt and padding are always 0.

In [ ]:
# TODO: from starter import tokenize_preference_pair, collate_preference_batch
# TODO: tokenize one pair; render a table of (position, token_id, token_str, response_mask)
# TODO: hand-verify: mask==0 over all prompt positions and all pad positions

In [ ]:
# TODO: build the paired DataLoader over the Lab 02 subset
# TODO: pull one batch; assert chosen/rejected share identical prompt prefixes

## 2. sequence_logprob — verify before you trust it

Required check: your sum-of-masked-per-token-log-probs must match an equivalent
`F.cross_entropy(reduction='none')` computation on a small batch.

In [ ]:
# TODO: implement sequence_logprob in starter.py, then:
# TODO: random-logits toy case -> compare against F.cross_entropy path
# TODO: real model + tokenizer case -> verify masking ignores prompt positions
# assert torch.allclose(mine, reference_check, atol=1e-5), 'verification failed'

## 3. The DPO objective — derive, then implement

Derive the loss from Bradley-Terry reparameterization ON PAPER first (Ch. 8).
Then fill `dpo_loss` in starter.py. Four forward passes feed it:
chosen/rejected x policy/reference. No packaged sigmoid-loss helpers.

In [ ]:
# TODO: forward policy on chosen/rejected -> sequence_logprob twice
# TODO: frozen reference forwards (forward_reference_model) -> two more logps
# TODO: call dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=CONFIG['beta'])
# TODO: loss.backward() smoke test on CPU-tiny model before any real run

## 4. Metrics & observability

Nothing counts as progress unless these move: implicit rewards (chosen/rejected),
margin, pairwise accuracy, response length. Appended to metrics.jsonl each step.

In [ ]:
# TODO: implement compute_metrics in starter.py
# TODO: print a first-step metric snapshot BEFORE any optimizer step (step-0 baseline)

## 5. Training run (prototype scale)

Small-run sanity check only; full runs go to the A40 via the script export.

In [ ]:
# TODO: tiny slice (~64 pairs), OLMo-2-1B-SFT or Qwen3-0.6B for Colab
# TODO: train loop with grad accumulation + clip-grad-norm, logging every log_every steps

## 6. β sweep analysis ({0.05, 0.1, 0.5})

Plot margin growth vs mean |log-ratio| drift per β arm. Artifacts from all three
runs come back here for plotting regardless of platform (cluster runs write JSONL).

In [ ]:
# TODO: load metrics.jsonl from the three arms
# TODO: plot margin curves overlayed by beta
# TODO: plot mean |log-ratio| curves overlayed by beta

## 7. Before/after generation panel

Fixed eval-prompt panel: step-0 (reference) vs final policy generations.

In [ ]:
# TODO: sample generations on eval_prompts_path at init and end of training
# TODO: print side-by-side transcripts for manual review

## Debugging journal

- Symptom: accuracy → 1.0 within ~50 steps + degraded generations — diagnose before continuing.
- Symptom: margin grows while chosen implicit reward stays negative — interpret.
(Record what you find in this notebook as you work.)